# AutoDIAL Full-Mamba HADA — Zhou2016 LOSO (Colab)
Enable a **T4 GPU**, then run the cell below. The Zhou2016 cache, logs, and result archive are persisted in Google Drive.

In [ ]:
# One-cell Colab: Runtime > Change runtime type > T4 GPU, then run.
import codecs
import os
import re
import select
import shutil
import subprocess
import sys
import time
from pathlib import Path
from google.colab import drive

# ---------------- Job configuration ----------------
# 'autodial' = learned source/target statistic mixing (main method);
# 'none'     = original HADA without AutoDIAL, for a same-branch baseline.
FEATURE_ALIGNMENT = 'autodial'
TARGET_SUBJECTS = list(range(1, 5))
JOB_TAG = 's01_s04'
DATASET = 'zhou2016'
BRANCH = 'feature/hada-full-mamba-autodial'
REPO_URL = 'https://github.com/CuongDM1806/tcformer-test.git'
RUN_TAG = f'{DATASET}_{FEATURE_ALIGNMENT}_{JOB_TAG}'
REPO_PATH = Path(f'/content/tcformer-autodial-{DATASET}')

# Mount Drive once and keep downloads/results across Colab reconnects.
if Path('/content/drive/MyDrive').is_dir():
    DRIVE_ROOT = Path('/content/drive')
else:
    mountpoint = Path('/content/drive')
    if mountpoint.exists() and any(mountpoint.iterdir()):
        mountpoint = Path('/content/google_drive')
    drive.mount(str(mountpoint))
    DRIVE_ROOT = mountpoint

MNE_DATA = DRIVE_ROOT / 'MyDrive/datasets/Zhou2016_MNE'
OUTPUT_DIR = DRIVE_ROOT / 'MyDrive/TCFormer-results/autodial'
TRAIN_LOG = OUTPUT_DIR / f'autodial_{RUN_TAG}.log'
RESULT_ARCHIVE = OUTPUT_DIR / f'autodial_{RUN_TAG}_results'


def run(command, cwd=None, env=None, stream=False, log_path=None):
    command = list(map(str, command))
    print('+', ' '.join(command), flush=True)
    if not stream:
        subprocess.run(
            command,
            cwd=str(cwd) if cwd else None,
            env=env,
            check=True,
        )
        return

    process = subprocess.Popen(
        command,
        cwd=str(cwd) if cwd else None,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        bufsize=0,
    )
    decoder = codecs.getincrementaldecoder('utf-8')(errors='replace')
    started_at = time.monotonic()
    with open(log_path, 'w', encoding='utf-8') as log_file:
        while True:
            ready, _, _ = select.select([process.stdout], [], [], 30)
            if ready:
                chunk = os.read(process.stdout.fileno(), 4096)
                if not chunk:
                    break
                output = decoder.decode(chunk)
                print(output, end='', flush=True)
                log_file.write(output)
                log_file.flush()
            elif process.poll() is None:
                elapsed = (time.monotonic() - started_at) / 60
                heartbeat = f'[Colab heartbeat] running | elapsed={elapsed:.1f}m\n'
                print(heartbeat, end='', flush=True)
                log_file.write(heartbeat)
                log_file.flush()
            else:
                break
        remaining = decoder.decode(b'', final=True)
        if remaining:
            print(remaining, end='', flush=True)
            log_file.write(remaining)

    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f'Training exited with code {return_code}. Full log: {log_path}'
        )


# ---------------- Clone branch and verify implementation ----------------
run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
if (REPO_PATH / '.git').is_dir():
    run(['git', 'remote', 'set-url', 'origin', REPO_URL], cwd=REPO_PATH)
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_PATH)
    run(['git', 'checkout', '--force', BRANCH], cwd=REPO_PATH)
    run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO_PATH)
else:
    run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, REPO_PATH])
run(['git', 'log', '-1', '--oneline'], cwd=REPO_PATH)

pipeline_text = (REPO_PATH / 'train_pipeline.py').read_text(encoding='utf-8')
model_text = (REPO_PATH / 'models/hada_tcformer.py').read_text(encoding='utf-8')
datamodule_text = (REPO_PATH / 'utils/get_datamodule_cls.py').read_text(encoding='utf-8')
assert 'ModelCheckpoint' in pipeline_text and 'monitor="val_acc"' in pipeline_text
assert 'class AutoDIALFeatureNorm' in model_text
assert 'zhou2016_loso' in datamodule_text
print('Verified: AutoDIAL model, Zhou2016 support, best source-validation checkpoint.')

UV = shutil.which('uv') or 'uv'
run([UV, 'venv', '--clear', '--python', '3.10', '.venv'], cwd=REPO_PATH)
PYTHON = REPO_PATH / '.venv/bin/python'
run([UV, 'pip', 'install', '--python', PYTHON, 'torch==2.7.1', 'torchvision==0.22.1',
     '--index-url', 'https://download.pytorch.org/whl/cu126'])
run([UV, 'pip', 'install', '--python', PYTHON, '-r', 'requirements.txt'], cwd=REPO_PATH)

# ---------------- Config override ----------------
override = (
    'import yaml; from pathlib import Path; '
    "p=Path('configs/hada_tcformer.yaml'); c=yaml.safe_load(p.read_text()); "
    f"c['subject_ids']={TARGET_SUBJECTS!r}; "
    "c['log_every_n_epochs']=10; "
    "c['enable_progress_bar']=False; "
    "c['model_kwargs']['log_every_n_batches']=0; "
    f"c['model_kwargs']['feature_alignment']={FEATURE_ALIGNMENT!r}; "
    "p.write_text(yaml.safe_dump(c, sort_keys=False))"
)
run([PYTHON, '-c', override], cwd=REPO_PATH)
run(['nvidia-smi'])
run([PYTHON, '-c', "import torch; print('PyTorch:', torch.__version__); "
     "print('CUDA:', torch.cuda.is_available()); "
     "assert torch.cuda.is_available(), 'Enable a Colab GPU accelerator'; "
     "print('GPU:', torch.cuda.get_device_name(0))"])

# ---------------- Train / test ----------------
MNE_DATA.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results_dir = REPO_PATH / 'results'
if results_dir.exists():
    shutil.rmtree(results_dir)
environment = os.environ.copy()
environment.update({
    'PYTHONUNBUFFERED': '1',
    'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True',
    'MPLBACKEND': 'Agg',
    'MNE_DATA': str(MNE_DATA),
    'MNE_DATASETS_ZHOU2016_PATH': str(MNE_DATA),
})
print(f'===== COLAB AUTODIAL FULL-MAMBA HADA | ZHOU2016 | 4 SUBJECTS | 3 CLASSES | 14 CH | 5.0 s | alignment={FEATURE_ALIGNMENT} | TARGETS={TARGET_SUBJECTS} =====', flush=True)
print('Protocol: source sessions 1-2=train, source session 3=best-checkpoint validation; target sessions 1-2=unlabeled UDA, target session 3=IM-TTA + AutoDIAL recalibration + primary test.', flush=True)
print('Live Drive log:', TRAIN_LOG, flush=True)
run([PYTHON, '-u', 'train_pipeline.py', '--model', 'hada_tcformer', '--dataset', DATASET,
     '--loso', '--gpu_id', '0'], cwd=REPO_PATH, env=environment, stream=True,
    log_path=TRAIN_LOG)

# ---------------- Summary and Drive export ----------------
log_text = TRAIN_LOG.read_text(encoding='utf-8', errors='replace')
alphas = re.findall(r'AutoDIAL alpha per feature group: (\[[^\]]*\])', log_text)
for subject_id, alpha in zip(TARGET_SUBJECTS, alphas):
    print(f'S{subject_id:02d} final alpha (CNN groups 1-3, SSM group): {alpha}')
archive = shutil.make_archive(str(RESULT_ARCHIVE), 'zip', root_dir=REPO_PATH, base_dir='results')
print('Completed targets:', TARGET_SUBJECTS)
print('Drive result archive:', archive)
print('Drive training log:', TRAIN_LOG)
